In [67]:
import sys
from pathlib import Path
# Si tu notebook está en la raíz, usá Path(".").resolve()
ROOT_DIR = Path("..").resolve()

# 2. Agregar la raíz al sistema de Python si no está cargada
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

import os    
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from src.preprocessing.common import check_time, check_time_difference
from src.paths import INTERIM_DIR
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# <center> 1. Carguemos los datos formateados

In [2]:
print(os.listdir(INTERIM_DIR))

['auh_edad_titular.csv', 'auh_valor_general.csv', 'cba.csv', 'cbt.csv', 'engho_gastos.csv', 'engho_hogares.csv', 'engho_personas.csv', 'eph_ripte.csv', 'eph_smvm.csv', 'ipc_canastas_cuyo.csv', 'ipc_canastas_gba.csv', 'ipc_canastas_nea.csv', 'ipc_canastas_noa.csv', 'ipc_canastas_pampeana.csv', 'ipc_canastas_patagonia.csv', 'ipc_general.csv']


In [3]:
dataframes = {} 
for name in os.listdir(INTERIM_DIR):
    # Validar que sea un archivo CSV
    if name.endswith(".csv"):
        # Quitar la extensión .csv para la clave (ej: "cba.csv" -> "cba")
        nombre_clave = name.replace(".csv", "")

        # Construir la ruta completa al archivo
        ruta_completa = os.path.join(INTERIM_DIR, name)

        # Guardar en el diccionario con el prefijo df_
        dataframes["df_" + nombre_clave] = pd.read_csv(ruta_completa, low_memory=False)

# Ahora puedes acceder directamente a tu DataFrame
for clave, valor in dataframes.items():
    globals()[clave] = valor

# <center> 2. Limpieza CBT-CBA</center>

### Cambiamos los tipos incorrectos.

In [19]:
dataframes['df_cba']["indice_tiempo"] = pd.to_datetime(dataframes['df_cba']["indice_tiempo"])
dataframes['df_cbt']["indice_tiempo"] = pd.to_datetime(dataframes['df_cbt']["indice_tiempo"])

### Veamos los valores atipicos

In [42]:
# fig funciona correctamente con Plotly Express
fig = px.line(dataframes['df_cbt']['gran_buenos_aires'])

fig1 = go.Figure(
    data=[
        go.Scatter(
            x=dataframes['df_cbt']['indice_tiempo'],  # Usamos la columna de tiempo para el eje X
            y=dataframes['df_cbt'][column], 
            mode='lines', 
            name=column
        )
        for column in dataframes['df_cbt'].columns 
        if column != 'indice_tiempo'  # Excluye la columna de tiempo de las líneas
    ],
    layout={"xaxis": {"title": "Tiempo"}, "yaxis": {"title": "Valores"}, "title": "CBT por Región"}
)

fig1.show()

### Veamos para CBA

In [52]:
# fig funciona correctamente con Plotly Express
fig = px.line(dataframes['df_cba']['gran_buenos_aires'])

fig1 = go.Figure(
    data=[
        go.Scatter(
            x=dataframes['df_cba']['indice_tiempo'],  # Usamos la columna de tiempo para el eje X
            y=dataframes['df_cba'][column], 
            mode='lines', 
            name=column
        )
        for column in dataframes['df_cba'].columns 
        if column != 'indice_tiempo'  # Excluye la columna de tiempo de las líneas
    ],
    layout={"xaxis": {"title": "Tiempo"}, "yaxis": {"title": "Valores"}, "title": "CBA por Región"}
)

fig1.show()

#### La inspección visual de las series temporales no evidencia saltos abruptos ni valores inconsistentes. La evolución de la CBT y CBA es continua para todas las regiones.

### Veamos ahora para la columna de indice_tiempo para ambos datasets:

In [85]:
minimo_CBT, maximo_CBT = check_time(dataframes['df_cbt'],'indice_tiempo') 
print(f"La minima entrada es: {minimo_CBT} La maxima entrada es: {maximo_CBT} para CBT")

La minima entrada es: 2016-04-01 00:00:00 La maxima entrada es: 2025-12-01 00:00:00 para CBT


In [86]:
minimo_CBA, maximo_CBA = check_time(dataframes['df_cba'],'indice_tiempo') 
print(f"La minima entrada es: {minimo_CBA} La maxima entrada es: {maximo_CBA} para CBA")

La minima entrada es: 2016-04-01 00:00:00 La maxima entrada es: 2025-12-01 00:00:00 para CBA


#### Vemos que para ambos datasets los minimos y maximos estan correctos.

## Contamos la cantidad de dias que separan a cada entrada:

In [87]:
check_time_difference(dataframes['df_cbt'],'indice_tiempo')

Distancias detectadas entre registros (en días):
31 days    67
30 days    40
28 days     7
29 days     2
Name: indice_tiempo, dtype: int64


In [88]:
check_time_difference(dataframes['df_cba'],'indice_tiempo')

Distancias detectadas entre registros (en días):
31 days    67
30 days    40
28 days     7
29 days     2
Name: indice_tiempo, dtype: int64


#### Como no tenemos mas de 31 dias de diferencia entre meses, vemos que no existen ningun mes faltante entre medio

# <center> Limpieza AUH </center>

In [106]:
dataframes["df_auh_edad_titular"].columns
columns_to_float = (dataframes["df_auh_edad_titular"].columns)[1:]
columns_to_float
# Asignamos el tipo corecto:
dataframes['df_auh_edad_titular'][columns_to_float] = dataframes['df_auh_edad_titular'][columns_to_float].astype(float)

# Asignamos a la columna Periodo, el tipo correcto:
dataframes['df_auh_edad_titular']['Periodo'] = pd.to_datetime(dataframes['df_auh_edad_titular']['Periodo'])


In [107]:
dataframes['df_auh_edad_titular'].dtypes

Periodo          datetime64[ns]
15 - 19                 float64
20 - 24                 float64
25 - 29                 float64
30 - 34                 float64
35 - 39                 float64
40 - 44                 float64
45 - 49                 float64
50 - 54                 float64
55 - 59                 float64
60 - 64                 float64
65 - 69                 float64
Más de 70               float64
Sin datos               float64
Total                   float64
Edad Promedio           float64
dtype: object

In [108]:
minimo_AUH_time, maximo_AUH_time = check_time(dataframes['df_auh_edad_titular'],'Periodo') 
print(f"La minima entrada es: {minimo_AUH_time} La maxima entrada es: {maximo_AUH_time}")

La minima entrada es: 2016-12-08 00:00:00 La maxima entrada es: 2026-02-01 00:00:00


In [109]:
check_time_difference(dataframes['df_auh_edad_titular'],'Periodo')

Distancias detectadas entre registros (en días):
31 days     16
30 days      8
360 days     2
365 days     2
370 days     1
358 days     1
371 days     1
29 days      1
28 days      1
Name: Periodo, dtype: int64


### Vemos que tenemos periodos faltantes.